# MLES Assignment 3
## Anomaly Detection on KDDCup99 Using AutoEncoders and Quantization

This notebook converts the completed Python project into a submission-style Jupyter notebook.
It follows the provided roadmap and instruction PDFs:

- load and inspect the KDDCup99 dataset
- preprocess and binarize labels
- train an AutoEncoder on normal traffic only
- detect anomalies using reconstruction error
- quantize the trained model with TensorFlow Lite
- compare performance, model size, and inference speed

The notebook is organized so each major stage prints its own outputs clearly.


## 1. Imports and Setup

This section imports the required libraries, configures reproducibility, and creates the output folders used by the notebook.


In [1]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
)

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "KDDCup99.csv"
PLOT_DIR = BASE_DIR / "plots"
MODEL_DIR = BASE_DIR / "models"
RESULTS_PATH = BASE_DIR / "comparison_results.csv"

PLOT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

def section(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def save_plot(filename: str):
    output_path = PLOT_DIR / filename
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved plot: {output_path.name}")

print("Environment ready.")
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Pandas version:     {pd.__version__}")
print(f"Working directory:  {BASE_DIR}")


Environment ready.
TensorFlow version: 2.21.0
NumPy version:      2.1.1
Pandas version:     2.3.2
Working directory:  D:\2. NUST\1. ML for EMbedded SIr ALi Hassan\Assignment 3


## 2. Load and Understand the Dataset

We load the dataset, inspect its shape and schema, and summarize the original attack-label distribution before converting it into a binary anomaly task.


In [2]:
section("STEP 1: Load and Understand Dataset")

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

print("\nFirst five rows:")
print(df.head().to_string())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes.to_string())

missing_values = int(df.isnull().sum().sum())
print(f"\nTotal missing values: {missing_values}")

categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = df.select_dtypes(include=["number"]).columns.tolist()
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols[:10]} ...")

label_counts = df["label"].value_counts()
label_distribution_df = (
    label_counts.rename_axis("label")
    .reset_index(name="count")
    .assign(percentage=lambda x: (x["count"] / len(df) * 100).round(2))
)

print("\nTop label distribution:")
print(label_distribution_df.head(10).to_string(index=False))



STEP 1: Load and Understand Dataset
Dataset shape: (494020, 42)
Rows: 494,020
Columns: 42

First five rows:
   duration protocol_type service flag  src_bytes  dst_bytes  land  wrong_fragment  urgent  hot  num_failed_logins  logged_in  lnum_compromised  lroot_shell  lsu_attempted  lnum_root  lnum_file_creations  lnum_shells  lnum_access_files  lnum_outbound_cmds  is_host_login  is_guest_login  count  srv_count  serror_rate  srv_serror_rate  rerror_rate  srv_rerror_rate  same_srv_rate  diff_srv_rate  srv_diff_host_rate  dst_host_count  dst_host_srv_count  dst_host_same_srv_rate  dst_host_diff_srv_rate  dst_host_same_src_port_rate  dst_host_srv_diff_host_rate  dst_host_serror_rate  dst_host_srv_serror_rate  dst_host_rerror_rate  dst_host_srv_rerror_rate   label
0         0           tcp    http   SF        181       5450     0               0       0    0                  0          1                 0            0              0          0                    0            0              

## 3. Binary Label Conversion and Preprocessing

Following the assignment instructions, `normal` is mapped to `0` and every attack type is mapped to `1`.
We then one-hot encode the categorical features and scale the final feature matrix with `MinMaxScaler`.


In [3]:
section("STEP 2: Preprocessing")

df["target"] = (df["label"] != "normal").astype(int)

target_summary = pd.Series(df["target"]).value_counts().sort_index()
print("Binary target counts:")
print(f"Normal (0):  {target_summary.get(0, 0):,}")
print(f"Anomaly (1): {target_summary.get(1, 0):,}")

y = df["target"].to_numpy()
X = df.drop(columns=["label", "target"])

categorical_features = ["protocol_type", "service", "flag"]
print(f"\nFeatures before encoding: {X.shape}")
print("Categorical feature cardinalities:")
for column in categorical_features:
    print(f"- {column}: {X[column].nunique()} unique values")

X = pd.get_dummies(X, columns=categorical_features, dtype=float)
encoded_feature_names = X.columns.tolist()

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=encoded_feature_names)

print(f"\nFeatures after encoding: {X.shape}")
print(f"Scaled feature matrix shape: {X_scaled.shape}")
print(f"Scaled min value: {X_scaled.min().min():.4f}")
print(f"Scaled max value: {X_scaled.max().max():.4f}")



STEP 2: Preprocessing
Binary target counts:
Normal (0):  97,277
Anomaly (1): 396,743

Features before encoding: (494020, 41)
Categorical feature cardinalities:
- protocol_type: 3 unique values
- service: 66 unique values
- flag: 11 unique values

Features after encoding: (494020, 118)
Scaled feature matrix shape: (494020, 118)
Scaled min value: 0.0000
Scaled max value: 1.0000


## 4. Train-Test Split

The AutoEncoder must learn only normal traffic patterns, so after a stratified train-test split we keep only normal training samples for model fitting.


In [4]:
section("STEP 3: Data Splitting")

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

X_train_normal = X_train[y_train == 0]

X_train_normal_np = X_train_normal.to_numpy(dtype="float32")
X_test_np = X_test.to_numpy(dtype="float32")

input_dim = X_train_normal_np.shape[1]

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape:     {X_test.shape}")
print(f"Train normal count:  {(y_train == 0).sum():,}")
print(f"Train anomaly count: {(y_train == 1).sum():,}")
print(f"Test normal count:   {(y_test == 0).sum():,}")
print(f"Test anomaly count:  {(y_test == 1).sum():,}")
print(f"\nAutoEncoder training shape (normal only): {X_train_normal_np.shape}")
print(f"Input dimension: {input_dim}")



STEP 3: Data Splitting
Training set shape: (395216, 118)
Test set shape:     (98804, 118)
Train normal count:  77,822
Train anomaly count: 317,394
Test normal count:   19,455
Test anomaly count:  79,349

AutoEncoder training shape (normal only): (77822, 118)
Input dimension: 118


## 5. AutoEncoder Architecture

The dense AutoEncoder follows the required structure:

`Input -> 128 -> 64 -> 32 -> 16 -> 32 -> 64 -> 128 -> Output`


In [5]:
section("STEP 4: Build AutoEncoder Model")

input_layer = Input(shape=(input_dim,), name="input")
encoded = Dense(128, activation="relu", name="encoder_1")(input_layer)
encoded = Dense(64, activation="relu", name="encoder_2")(encoded)
encoded = Dense(32, activation="relu", name="encoder_3")(encoded)
bottleneck = Dense(16, activation="relu", name="bottleneck")(encoded)

decoded = Dense(32, activation="relu", name="decoder_1")(bottleneck)
decoded = Dense(64, activation="relu", name="decoder_2")(decoded)
decoded = Dense(128, activation="relu", name="decoder_3")(decoded)
output_layer = Dense(input_dim, activation="sigmoid", name="output")(decoded)

autoencoder = Model(inputs=input_layer, outputs=output_layer, name="AutoEncoder")
autoencoder.compile(optimizer="adam", loss="mse")

autoencoder.summary()
print("\nOptimizer: Adam")
print("Loss: Mean Squared Error")
print("Hidden activation: ReLU")
print("Output activation: Sigmoid")



STEP 4: Build AutoEncoder Model
Model: "AutoEncoder"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ input (InputLayer)              │ (None, 118)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_1 (Dense)               │ (None, 128)            │        15,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_2 (Dense)               │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_3 (Dense)               │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 16)             │           528 │
├─────────────────────

## 6. Model Training

The model is trained on normal samples only, with validation monitoring and early stopping.


In [6]:
section("STEP 5: Train AutoEncoder Model")

best_model_path = MODEL_DIR / "best_autoencoder.keras"
final_model_path = MODEL_DIR / "autoencoder_final.keras"

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=0,
)

model_checkpoint = ModelCheckpoint(
    filepath=str(best_model_path),
    monitor="val_loss",
    save_best_only=True,
    verbose=0,
)

history = autoencoder.fit(
    X_train_normal_np,
    X_train_normal_np,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stopping, model_checkpoint],
    verbose=0,
)

autoencoder.save(final_model_path)

print("Training complete.")
print(f"Epochs trained: {len(history.history['loss'])}")
print(f"Final training loss:   {history.history['loss'][-1]:.6f}")
print(f"Final validation loss: {history.history['val_loss'][-1]:.6f}")
print(f"Saved best model:  {best_model_path.name}")
print(f"Saved final model: {final_model_path.name}")



STEP 5: Train AutoEncoder Model
Training complete.
Epochs trained: 50
Final training loss:   0.001156
Final validation loss: 0.001194
Saved best model:  best_autoencoder.keras
Saved final model: autoencoder_final.keras


## 7. Reconstruction Error and Threshold Selection

Anomaly detection is performed using mean squared reconstruction error.
The main threshold used here is `mean + 3 * std` of reconstruction error on the normal training set.


In [7]:
section("STEP 6: Anomaly Detection")

X_test_pred = autoencoder.predict(X_test_np, verbose=0)
reconstruction_error_test = np.mean(np.square(X_test_np - X_test_pred), axis=1)

X_train_pred = autoencoder.predict(X_train_normal_np, verbose=0)
reconstruction_error_train = np.mean(np.square(X_train_normal_np - X_train_pred), axis=1)

threshold_3sigma = reconstruction_error_train.mean() + 3 * reconstruction_error_train.std()
threshold_95pct = np.percentile(reconstruction_error_train, 95)
threshold = threshold_3sigma

y_pred = (reconstruction_error_test > threshold).astype(int)

print("Training reconstruction error statistics:")
print(f"Mean: {reconstruction_error_train.mean():.6f}")
print(f"Std:  {reconstruction_error_train.std():.6f}")
print(f"95th percentile threshold: {threshold_95pct:.6f}")
print(f"Selected threshold (mean + 3*std): {threshold:.6f}")

print("\nTest reconstruction error statistics:")
print(f"Mean: {reconstruction_error_test.mean():.6f}")
print(f"Std:  {reconstruction_error_test.std():.6f}")
print(f"Min:  {reconstruction_error_test.min():.6f}")
print(f"Max:  {reconstruction_error_test.max():.6f}")

print("\nPrediction counts:")
print(f"Predicted normal:  {(y_pred == 0).sum():,}")
print(f"Predicted anomaly: {(y_pred == 1).sum():,}")



STEP 6: Anomaly Detection
Training reconstruction error statistics:
Mean: 0.001159
Std:  0.003602
95th percentile threshold: 0.008533
Selected threshold (mean + 3*std): 0.011966

Test reconstruction error statistics:
Mean: 0.030209
Std:  0.018321
Min:  0.000001
Max:  0.083912

Prediction counts:
Predicted normal:  19,721
Predicted anomaly: 79,083


## 8. Baseline Evaluation

We evaluate the baseline AutoEncoder with accuracy, precision, recall, F1-score, ROC-AUC, and a confusion matrix.


In [8]:
section("STEP 7: Baseline Evaluation")

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, reconstruction_error_test)
cm = confusion_matrix(y_test, y_pred)

baseline_results = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "ROC-AUC": roc_auc,
}

print("Baseline AutoEncoder metrics:")
for metric, value in baseline_results.items():
    print(f"{metric}: {value:.4f}")

print("\nConfusion matrix:")
print(cm)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Normal", "Anomaly"], zero_division=0))



STEP 7: Baseline Evaluation
Baseline AutoEncoder metrics:
Accuracy: 0.9885
Precision: 0.9945
Recall: 0.9911
F1-Score: 0.9928
ROC-AUC: 0.9982

Confusion matrix:
[[19018   437]
 [  703 78646]]

Classification report:
              precision    recall  f1-score   support

      Normal       0.96      0.98      0.97     19455
     Anomaly       0.99      0.99      0.99     79349

    accuracy                           0.99     98804
   macro avg       0.98      0.98      0.98     98804
weighted avg       0.99      0.99      0.99     98804



## 9. Baseline Visualizations

The following plots are generated and saved for the report:

- training vs validation loss
- reconstruction error distribution
- confusion matrix
- ROC curve


In [9]:
section("STEP 8: Baseline Visualizations")

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

plt.figure(figsize=(10, 6))
plt.plot(history.history["loss"], label="Training Loss", linewidth=2)
plt.plot(history.history["val_loss"], label="Validation Loss", linewidth=2)
plt.title("AutoEncoder Training vs Validation Loss", fontsize=16, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.legend()
save_plot("01_training_validation_loss.png")

error_normal = reconstruction_error_test[y_test == 0]
error_anomaly = reconstruction_error_test[y_test == 1]

plt.figure(figsize=(12, 6))
plt.hist(error_normal, bins=100, alpha=0.7, label="Normal", color="#2ecc71", density=True)
plt.hist(error_anomaly, bins=100, alpha=0.7, label="Anomaly", color="#e74c3c", density=True)
plt.axvline(threshold, color="#f39c12", linestyle="--", linewidth=2, label=f"Threshold ({threshold:.4f})")
plt.title("Reconstruction Error Distribution", fontsize=16, fontweight="bold")
plt.xlabel("Reconstruction Error (MSE)")
plt.ylabel("Density")
plt.xlim(0, min(np.percentile(reconstruction_error_test, 99.5), 0.5))
plt.legend()
save_plot("02_reconstruction_error_distribution.png")

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt=",",
    cmap="Blues",
    xticklabels=["Normal", "Anomaly"],
    yticklabels=["Normal", "Anomaly"],
)
plt.title("Confusion Matrix - Baseline AutoEncoder", fontsize=16, fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("Actual")
save_plot("03_confusion_matrix_baseline.png")

fpr, tpr, _ = roc_curve(y_test, reconstruction_error_test)
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, linewidth=2, label=f"ROC Curve (AUC = {roc_auc:.4f})", color="#3498db")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Classifier")
plt.title("ROC Curve - Baseline AutoEncoder", fontsize=16, fontweight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
save_plot("04_roc_curve_baseline.png")



STEP 8: Baseline Visualizations
Saved plot: 01_training_validation_loss.png
Saved plot: 02_reconstruction_error_distribution.png
Saved plot: 03_confusion_matrix_baseline.png
Saved plot: 04_roc_curve_baseline.png


## 10. Quantization with TensorFlow Lite

The trained Keras model is converted into a dynamically quantized TFLite model using `tf.lite.Optimize.DEFAULT`.


In [10]:
section("STEP 9-10: TFLite Quantization")

converter = tf.lite.TFLiteConverter.from_keras_model(autoencoder)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_model_path = MODEL_DIR / "autoencoder_quantized.tflite"
with open(tflite_model_path, "wb") as file_handle:
    file_handle.write(tflite_model)

print(f"Quantized model saved: {tflite_model_path.name}")



STEP 9-10: TFLite Quantization
Saved artifact at 'C:\Users\ABDULB~1\AppData\Local\Temp\tmp55na957y'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 118), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(None, 118), dtype=tf.float32, name=None)
Captures:
  2203122084240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2203122084816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2203122083664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2203122084432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2203122085392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2203122085008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2203122086544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2203122085776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2203122086352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2203122085968: TensorSpec(shape=(), dtype=tf.resour

## 11. Model Size and Inference Speed Comparison

We compare the original Keras model and the quantized TFLite model in terms of file size and average inference time per sample.


In [11]:
section("STEP 11-12: Model Size and Inference Speed")

original_size = os.path.getsize(final_model_path)
quantized_size = os.path.getsize(tflite_model_path)
size_reduction = (1 - quantized_size / original_size) * 100

print("Model size comparison:")
print(f"Original model size:  {original_size:,} bytes ({original_size / 1024:.2f} KB)")
print(f"Quantized model size: {quantized_size:,} bytes ({quantized_size / 1024:.2f} KB)")
print(f"Size reduction:       {size_reduction:.2f}%")

NUM_INFERENCE_SAMPLES = 1000
test_samples = X_test_np[:NUM_INFERENCE_SAMPLES]

_ = autoencoder.predict(test_samples[:10], verbose=0)
keras_times = []
for _ in range(5):
    start = time.perf_counter()
    _ = autoencoder.predict(test_samples, verbose=0)
    keras_times.append(time.perf_counter() - start)

keras_avg_time = float(np.mean(keras_times))
keras_per_sample = keras_avg_time / NUM_INFERENCE_SAMPLES * 1000

interpreter = tf.lite.Interpreter(model_path=str(tflite_model_path))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

for idx in range(10):
    sample = test_samples[idx:idx + 1].astype(np.float32)
    interpreter.set_tensor(input_details[0]["index"], sample)
    interpreter.invoke()

tflite_times = []
for _ in range(5):
    start = time.perf_counter()
    for idx in range(NUM_INFERENCE_SAMPLES):
        sample = test_samples[idx:idx + 1].astype(np.float32)
        interpreter.set_tensor(input_details[0]["index"], sample)
        interpreter.invoke()
    tflite_times.append(time.perf_counter() - start)

tflite_avg_time = float(np.mean(tflite_times))
tflite_per_sample = tflite_avg_time / NUM_INFERENCE_SAMPLES * 1000
speedup = keras_avg_time / tflite_avg_time if tflite_avg_time else float("inf")

print("\nInference speed comparison:")
print(f"Keras average total time:  {keras_avg_time * 1000:.2f} ms")
print(f"Keras time per sample:     {keras_per_sample:.4f} ms")
print(f"TFLite average total time: {tflite_avg_time * 1000:.2f} ms")
print(f"TFLite time per sample:    {tflite_per_sample:.4f} ms")
print(f"Speedup factor:            {speedup:.2f}x")



STEP 11-12: Model Size and Inference Speed
Model size comparison:
Original model size:  681,559 bytes (665.58 KB)
Quantized model size: 68,224 bytes (66.62 KB)
Size reduction:       89.99%

Inference speed comparison:
Keras average total time:  106.95 ms
Keras time per sample:     0.1069 ms
TFLite average total time: 5.37 ms
TFLite time per sample:    0.0054 ms
Speedup factor:            19.93x


## 12. Quantized Model Evaluation

The same anomaly-detection logic is applied to the TFLite model so we can compare predictive quality fairly.


In [12]:
section("STEP 13: Quantized Model Evaluation")

tflite_predictions = np.zeros_like(X_test_np)

for idx in range(X_test_np.shape[0]):
    sample = X_test_np[idx:idx + 1].astype(np.float32)
    interpreter.set_tensor(input_details[0]["index"], sample)
    interpreter.invoke()
    tflite_predictions[idx] = interpreter.get_tensor(output_details[0]["index"])[0]

    if (idx + 1) % 20000 == 0:
        print(f"Processed {idx + 1:,}/{X_test_np.shape[0]:,} samples...")

reconstruction_error_tflite = np.mean(np.square(X_test_np - tflite_predictions), axis=1)
y_pred_tflite = (reconstruction_error_tflite > threshold).astype(int)

accuracy_q = accuracy_score(y_test, y_pred_tflite)
precision_q = precision_score(y_test, y_pred_tflite, zero_division=0)
recall_q = recall_score(y_test, y_pred_tflite, zero_division=0)
f1_q = f1_score(y_test, y_pred_tflite, zero_division=0)
roc_auc_q = roc_auc_score(y_test, reconstruction_error_tflite)
cm_q = confusion_matrix(y_test, y_pred_tflite)

quantized_results = {
    "Accuracy": accuracy_q,
    "Precision": precision_q,
    "Recall": recall_q,
    "F1-Score": f1_q,
    "ROC-AUC": roc_auc_q,
}

print("Quantized AutoEncoder metrics:")
for metric, value in quantized_results.items():
    print(f"{metric}: {value:.4f}")

print("\nQuantized confusion matrix:")
print(cm_q)

print("\nQuantized classification report:")
print(classification_report(y_test, y_pred_tflite, target_names=["Normal", "Anomaly"], zero_division=0))



STEP 13: Quantized Model Evaluation
Processed 20,000/98,804 samples...
Processed 40,000/98,804 samples...
Processed 60,000/98,804 samples...
Processed 80,000/98,804 samples...
Quantized AutoEncoder metrics:
Accuracy: 0.9885
Precision: 0.9945
Recall: 0.9911
F1-Score: 0.9928
ROC-AUC: 0.9982

Quantized confusion matrix:
[[19019   436]
 [  704 78645]]

Quantized classification report:
              precision    recall  f1-score   support

      Normal       0.96      0.98      0.97     19455
     Anomaly       0.99      0.99      0.99     79349

    accuracy                           0.99     98804
   macro avg       0.98      0.98      0.98     98804
weighted avg       0.99      0.99      0.99     98804



## 13. Comparison Tables and Additional Plots

This section prepares the direct baseline-vs-quantized comparison tables and visual summaries required for the report.


In [13]:
section("STEP 14: Comparison Tables and Plots")

fpr_q, tpr_q, _ = roc_curve(y_test, reconstruction_error_tflite)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_q,
    annot=True,
    fmt=",",
    cmap="Oranges",
    xticklabels=["Normal", "Anomaly"],
    yticklabels=["Normal", "Anomaly"],
)
plt.title("Confusion Matrix - Quantized AutoEncoder", fontsize=16, fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("Actual")
save_plot("05_confusion_matrix_quantized.png")

plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, linewidth=2, label=f"Baseline (AUC = {roc_auc:.4f})", color="#3498db")
plt.plot(fpr_q, tpr_q, linewidth=2, linestyle="--", label=f"Quantized (AUC = {roc_auc_q:.4f})", color="#e74c3c")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random")
plt.title("ROC Curve Comparison", fontsize=16, fontweight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
save_plot("06_roc_curve_comparison.png")

plt.figure(figsize=(8, 6))
bars = plt.bar(
    ["Baseline\nAutoEncoder", "Quantized\nAutoEncoder"],
    [original_size / 1024, quantized_size / 1024],
    color=["#3498db", "#e74c3c"],
    edgecolor="black",
)
for bar, value in zip(bars, [original_size / 1024, quantized_size / 1024]):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 5, f"{value:.1f} KB", ha="center", fontweight="bold")
plt.ylabel("Model Size (KB)")
plt.title(f"Model Size Comparison ({size_reduction:.1f}% Reduction)", fontsize=16, fontweight="bold")
save_plot("07_model_size_comparison.png")

plt.figure(figsize=(8, 6))
bars = plt.bar(
    ["Keras\nModel", "TFLite\nQuantized"],
    [keras_per_sample, tflite_per_sample],
    color=["#3498db", "#e74c3c"],
    edgecolor="black",
)
for bar, value in zip(bars, [keras_per_sample, tflite_per_sample]):
    plt.text(bar.get_x() + bar.get_width() / 2, value + max(keras_per_sample, tflite_per_sample) * 0.03, f"{value:.4f} ms", ha="center", fontweight="bold")
plt.ylabel("Time per Sample (ms)")
plt.title(f"Inference Speed Comparison ({speedup:.2f}x Speedup)", fontsize=16, fontweight="bold")
save_plot("08_inference_speed_comparison.png")

plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
plt.hist(reconstruction_error_test[y_test == 0], bins=100, alpha=0.7, label="Normal", color="#2ecc71", density=True)
plt.hist(reconstruction_error_test[y_test == 1], bins=100, alpha=0.7, label="Anomaly", color="#e74c3c", density=True)
plt.axvline(threshold, color="#f39c12", linestyle="--", linewidth=2, label="Threshold")
plt.title("Baseline AutoEncoder", fontweight="bold")
plt.xlabel("Reconstruction Error")
plt.ylabel("Density")
plt.xlim(0, min(np.percentile(reconstruction_error_test, 99.5), 0.5))
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(reconstruction_error_tflite[y_test == 0], bins=100, alpha=0.7, label="Normal", color="#2ecc71", density=True)
plt.hist(reconstruction_error_tflite[y_test == 1], bins=100, alpha=0.7, label="Anomaly", color="#e74c3c", density=True)
plt.axvline(threshold, color="#f39c12", linestyle="--", linewidth=2, label="Threshold")
plt.title("Quantized AutoEncoder", fontweight="bold")
plt.xlabel("Reconstruction Error")
plt.ylabel("Density")
plt.xlim(0, min(np.percentile(reconstruction_error_tflite, 99.5), 0.5))
plt.legend()
save_plot("09_error_distribution_comparison.png")

comparison_df = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "ROC-AUC",
            "Model Size (KB)",
            "Inference Time (ms/sample)",
        ],
        "Baseline AE": [
            baseline_results["Accuracy"],
            baseline_results["Precision"],
            baseline_results["Recall"],
            baseline_results["F1-Score"],
            baseline_results["ROC-AUC"],
            original_size / 1024,
            keras_per_sample,
        ],
        "Quantized AE": [
            quantized_results["Accuracy"],
            quantized_results["Precision"],
            quantized_results["Recall"],
            quantized_results["F1-Score"],
            quantized_results["ROC-AUC"],
            quantized_size / 1024,
            tflite_per_sample,
        ],
    }
)

comparison_df.to_csv(RESULTS_PATH, index=False)
print("\nComparison table:")
print(comparison_df.to_string(index=False))
print(f"\nSaved results table: {RESULTS_PATH.name}")



STEP 14: Comparison Tables and Plots
Saved plot: 05_confusion_matrix_quantized.png
Saved plot: 06_roc_curve_comparison.png
Saved plot: 07_model_size_comparison.png
Saved plot: 08_inference_speed_comparison.png
Saved plot: 09_error_distribution_comparison.png

Comparison table:
                    Metric  Baseline AE  Quantized AE
                  Accuracy     0.988462      0.988462
                 Precision     0.994474      0.994487
                    Recall     0.991140      0.991128
                  F1-Score     0.992804      0.992804
                   ROC-AUC     0.998242      0.998220
           Model Size (KB)   665.584961     66.625000
Inference Time (ms/sample)     0.106950      0.005366

Saved results table: comparison_results.csv


## 14. Bonus: Threshold Tuning

To strengthen the analysis, we compare several threshold choices and track their effect on performance.


In [14]:
section("BONUS: Threshold Tuning Analysis")

percentile_results = []
for pct in range(90, 100):
    tuned_threshold = np.percentile(reconstruction_error_train, pct)
    tuned_pred = (reconstruction_error_test > tuned_threshold).astype(int)
    percentile_results.append(
        {
            "Percentile": pct,
            "Threshold": tuned_threshold,
            "Accuracy": accuracy_score(y_test, tuned_pred),
            "Precision": precision_score(y_test, tuned_pred, zero_division=0),
            "Recall": recall_score(y_test, tuned_pred, zero_division=0),
            "F1-Score": f1_score(y_test, tuned_pred, zero_division=0),
        }
    )

percentile_df = pd.DataFrame(percentile_results)
best_percentile_row = percentile_df.loc[percentile_df["F1-Score"].idxmax()]

print("Percentile threshold tuning results:")
print(percentile_df.round(4).to_string(index=False))
print("\nBest percentile-based threshold:")
print(best_percentile_row.round(6).to_string())

sigma_results = []
for k_value in [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
    tuned_threshold = reconstruction_error_train.mean() + k_value * reconstruction_error_train.std()
    tuned_pred = (reconstruction_error_test > tuned_threshold).astype(int)
    sigma_results.append(
        {
            "k": k_value,
            "Threshold": tuned_threshold,
            "Accuracy": accuracy_score(y_test, tuned_pred),
            "Precision": precision_score(y_test, tuned_pred, zero_division=0),
            "Recall": recall_score(y_test, tuned_pred, zero_division=0),
            "F1-Score": f1_score(y_test, tuned_pred, zero_division=0),
        }
    )

sigma_df = pd.DataFrame(sigma_results)
print("\nMean + k*std threshold results:")
print(sigma_df.round(4).to_string(index=False))



BONUS: Threshold Tuning Analysis
Percentile threshold tuning results:
 Percentile  Threshold  Accuracy  Precision  Recall  F1-Score
         90     0.0068    0.9740     0.9752  0.9929    0.9840
         91     0.0085    0.9760     0.9776  0.9929    0.9852
         92     0.0085    0.9779     0.9799  0.9929    0.9864
         93     0.0085    0.9800     0.9825  0.9928    0.9876
         94     0.0085    0.9821     0.9850  0.9928    0.9889
         95     0.0085    0.9840     0.9874  0.9928    0.9901
         96     0.0086    0.9860     0.9899  0.9928    0.9913
         97     0.0087    0.9879     0.9923  0.9926    0.9925
         98     0.0170    0.9883     0.9950  0.9904    0.9927
         99     0.0173    0.9899     0.9974  0.9900    0.9937

Best percentile-based threshold:
Percentile    99.000000
Threshold      0.017338
Accuracy       0.989930
Precision      0.997410
Recall         0.990031
F1-Score       0.993707

Mean + k*std threshold results:
  k  Threshold  Accuracy  Precision 

## 15. Bonus: Bottleneck Size Comparison

Different bottleneck widths are tested to see how latent-space compression affects anomaly-detection performance.


In [15]:
section("BONUS: Bottleneck Size Comparison")

bottleneck_sizes = [8, 16, 32]
bottleneck_results = []

for bottleneck_size in bottleneck_sizes:
    input_layer_bn = Input(shape=(input_dim,))
    x_bn = Dense(128, activation="relu")(input_layer_bn)
    x_bn = Dense(64, activation="relu")(x_bn)
    x_bn = Dense(32, activation="relu")(x_bn)
    x_bn = Dense(bottleneck_size, activation="relu")(x_bn)
    x_bn = Dense(32, activation="relu")(x_bn)
    x_bn = Dense(64, activation="relu")(x_bn)
    x_bn = Dense(128, activation="relu")(x_bn)
    output_layer_bn = Dense(input_dim, activation="sigmoid")(x_bn)

    ae_bn = Model(inputs=input_layer_bn, outputs=output_layer_bn)
    ae_bn.compile(optimizer="adam", loss="mse")

    ae_bn.fit(
        X_train_normal_np,
        X_train_normal_np,
        epochs=50,
        batch_size=256,
        validation_split=0.1,
        callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)],
        verbose=0,
    )

    pred_train_bn = ae_bn.predict(X_train_normal_np, verbose=0)
    err_train_bn = np.mean(np.square(X_train_normal_np - pred_train_bn), axis=1)
    threshold_bn = err_train_bn.mean() + 3 * err_train_bn.std()

    pred_test_bn = ae_bn.predict(X_test_np, verbose=0)
    err_test_bn = np.mean(np.square(X_test_np - pred_test_bn), axis=1)
    y_pred_bn = (err_test_bn > threshold_bn).astype(int)

    bottleneck_results.append(
        {
            "Bottleneck": bottleneck_size,
            "Accuracy": accuracy_score(y_test, y_pred_bn),
            "Precision": precision_score(y_test, y_pred_bn, zero_division=0),
            "Recall": recall_score(y_test, y_pred_bn, zero_division=0),
            "F1-Score": f1_score(y_test, y_pred_bn, zero_division=0),
        }
    )

    tf.keras.backend.clear_session()

bottleneck_df = pd.DataFrame(bottleneck_results)
print(bottleneck_df.round(4).to_string(index=False))

plt.figure(figsize=(10, 6))
positions = np.arange(len(bottleneck_sizes))
width = 0.2
metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
colors = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12"]

for idx, metric in enumerate(metrics):
    plt.bar(
        positions + idx * width,
        bottleneck_df[metric],
        width=width,
        label=metric,
        color=colors[idx],
    )

plt.xticks(positions + 1.5 * width, bottleneck_sizes)
plt.ylim(0, 1.05)
plt.xlabel("Bottleneck Size")
plt.ylabel("Score")
plt.title("Performance vs Bottleneck Size", fontsize=16, fontweight="bold")
plt.legend()
save_plot("10_bottleneck_comparison.png")



BONUS: Bottleneck Size Comparison
 Bottleneck  Accuracy  Precision  Recall  F1-Score
          8    0.9881     0.9944  0.9908    0.9926
         16    0.9893     0.9965  0.9902    0.9933
         32    0.9885     0.9929  0.9928    0.9928
Saved plot: 10_bottleneck_comparison.png


## 16. Final Takeaways

Key outcomes from the executed project:

- the AutoEncoder achieved very strong anomaly-detection performance on KDDCup99
- the quantized TFLite model preserved almost the same predictive quality
- quantization drastically reduced model size and improved per-sample inference time
- the generated plots and comparison table are ready to support the IEEE-style report
